In [ ]:
# Load required packages
library(Seurat)
library(pheatmap)
library(dplyr)
library(tidyr)
library(readxl)

# ========== Define common parameters ==========
target_genes <- c("JUN", "FGF2", "CD44", "FOS", "BRCA1", "EGFR", "CTNNB1", "STAT3", "MDM2", "SPP1", "TLR2", "HIF1A", "ITGAV", "IGF1", "PPARG", "GAPDH", "TGFB1", "STAT1", "ITGB1", "ACTB")
pathology_metrics <- c("Braak tangle stage", "ABC-A_AB", "ABC-B_NFT", "ABC-A_NP")

# ========== Part 1: Gene expression vs pathology correlation (snRNA-seq) ==========
# Extract normalized expression data for target genes
gene_expr <- GetAssayData(seurat_obj, assay = "RNA", slot = "data")
valid_genes <- intersect(target_genes, rownames(gene_expr))
target_expr <- gene_expr[valid_genes, ]

# Get pathology metrics from metadata
sample_info <- seurat_obj@meta.data[, pathology_metrics, drop = FALSE]
sample_info <- sample_info[colnames(target_expr), ]

# Initialize result matrices
cor_results <- matrix(NA, nrow = length(valid_genes), ncol = length(pathology_metrics),
                     dimnames = list(valid_genes, pathology_metrics))
pval_results <- matrix(NA, nrow = length(valid_genes), ncol = length(pathology_metrics),
                      dimnames = list(valid_genes, pathology_metrics))

# Compute Spearman correlation for each gene-metric pair
for (gene in valid_genes) {
  for (metric in pathology_metrics) {
    expr_data <- as.numeric(target_expr[gene, ])
    path_data <- as.numeric(sample_info[, metric])
    valid_idx <- !is.na(expr_data) & !is.na(path_data) & path_data != -9
    if (sum(valid_idx) > 3) {
      cor_test <- suppressWarnings(cor.test(expr_data[valid_idx], path_data[valid_idx], method = "spearman"))
      cor_results[gene, metric] <- cor_test$estimate
      pval_results[gene, metric] <- cor_test$p.value
    }
  }
}

# Multiple testing correction (BH)
adj_pval <- matrix(p.adjust(as.vector(pval_results), method = "BH"),
                   nrow = nrow(pval_results), ncol = ncol(pval_results),
                   dimnames = dimnames(pval_results))

# Significance symbols
sig_symbols <- matrix("", nrow = nrow(adj_pval), ncol = ncol(adj_pval))
sig_symbols[adj_pval < 0.05] <- "*"
sig_symbols[adj_pval < 0.01] <- "**"
sig_symbols[adj_pval < 0.001] <- "***"
rownames(sig_symbols) <- rownames(adj_pval)
colnames(sig_symbols) <- colnames(adj_pval)

# Heatmap color scale (blue-white-red, centered at 0)
max_cor <- max(abs(cor_results), na.rm = TRUE)
col_breaks <- seq(-max_cor, max_cor, length.out = 101)
col_fun <- colorRampPalette(c("blue", "white", "red"))(100)

# Draw and save heatmap
pheatmap(cor_results,
         color = col_fun,
         breaks = col_breaks,
         cluster_rows = FALSE,
         cluster_cols = TRUE,
         display_numbers = sig_symbols,
         number_color = "black",
         fontsize_number = 10,
         main = "Gene Expression - Pathology Correlation",
         filename = "gene_pathology_correlation.pdf",
         width = 8, height = 12, cellwidth = 20, cellheight = 15)

# ========== Part 2: Gene-wise Beta values from CSF proteomics ==========
# Read CSF protein data
df <- readxl::read_excel('/share/home/bgi_idbraorz/小鼠衰老单细胞/脑脊液蛋白指定基因数据2.xlsx')

# Ensure gene order matches target_genes
gene_order <- target_genes
df <- df %>% mutate(`Entrez gene symbol` = factor(`Entrez gene symbol`, levels = gene_order)) %>%
  arrange(`Entrez gene symbol`)

# Prepare heatmap matrix (Beta values)
heatmap_df <- df %>% select(`Entrez gene symbol`, Group, Beta) %>%
  pivot_wider(names_from = Group, values_from = Beta)
heatmap_matrix <- as.matrix(heatmap_df %>% select(-`Entrez gene symbol`))
rownames(heatmap_matrix) <- heatmap_df$`Entrez gene symbol`

# Significance symbols from P values
sig_df <- df %>% select(`Entrez gene symbol`, Group, P) %>%
  pivot_wider(names_from = Group, values_from = P)
sig_matrix <- as.matrix(sig_df %>% select(-`Entrez gene symbol`))
rownames(sig_matrix) <- sig_df$`Entrez gene symbol`

# Function to get significance markers
get_signif <- function(p) {
  ifelse(p < 0.001, '***', ifelse(p < 0.01, '**', ifelse(p < 0.05, '*', '')))
}
sig_symbols <- matrix(sapply(sig_matrix, get_signif), nrow = nrow(sig_matrix), dimnames = dimnames(sig_matrix))

# Color scale for Beta
max_beta <- max(abs(heatmap_matrix), na.rm = TRUE)
col_breaks <- seq(-max_beta, max_beta, length.out = 101)
col_fun <- colorRampPalette(c("blue", "white", "red"))(100)

# Draw heatmap
pheatmap(heatmap_matrix,
         color = col_fun,
         breaks = col_breaks,
         cluster_rows = FALSE,
         cluster_cols = FALSE,
         display_numbers = sig_symbols,
         number_color = "black",
         fontsize_number = 8,
         main = "Gene - Group Beta Heatmap (CSF Proteomics)",
         fontsize_row = 10,
         fontsize_col = 10,
         show_rownames = TRUE,
         filename = "gene_group_heatmap.pdf",
         width = 10, height = 12, cellwidth = 20, cellheight = 18)